In [ ]:
# !wget https://www.post.japanpost.jp/zipcode/dl/kogaki/zip/ken_all.zip

In [ ]:
# !unzip ken_all.zip

In [ ]:
import pandas as pd

filepath_jppost = 'KEN_ALL.CSV'

# Read jppost csv
jppost_org_df = pd.read_csv(filepath_jppost, encoding='shift-jis', header=None)
jppost_org_df.head()

In [ ]:
jppost_df = pd.DataFrame()

jppost_df['todofuken'] = jppost_org_df[6]
jppost_df['shikuchoson'] = jppost_org_df[7]
jppost_df['concated'] = jppost_org_df[6].str.cat(jppost_org_df[7])

jppost_df = jppost_df.drop_duplicates()
jppost_df = jppost_df.reset_index(drop=True)

jppost_df

In [ ]:
SIM_NAME = 'similarity'

def set_chimei_categories_list():
    chimei_categories_list = list()
    chimei_categories_list.append('00_prefectures')
    chimei_categories_list.append('01_shi')
    chimei_categories_list.append('02_ku')
    chimei_categories_list.append('03_gun')
    chimei_categories_list.append('04_cho')
    chimei_categories_list.append('05_son')
    
    return chimei_categories_list


def set_filepaths_input_dict(chimei_categories_list):
    filepaths_input_base = './chimei/'
    filepaths_input_dict = dict()

    for chimei_category in chimei_categories_list:
        filepaths_input = filepaths_input_base + chimei_category + '_list.txt'
        filepaths_input_dict[chimei_category] = filepaths_input
    
    return filepaths_input_dict
        

def set_chimeis_lists_dict(filepaths_input_dict):
    chimeis_lists_dict = dict()
    
    for chimei_category, filepath_input in filepaths_input_dict.items():
        chimeis_list = list()
        
        with open(filepath_input) as fp:
            for line in fp:
                chimeis_list.append(line.rstrip())
        
        chimeis_lists_dict[chimei_category] = chimeis_list
    
    return chimeis_lists_dict


# Create a chimei category list.
chimei_categories_list = set_chimei_categories_list()

# Create a dictionary of the paths of the input files by chimei category.
filepaths_input_dict = set_filepaths_input_dict(chimei_categories_list)

# Create a dictionary of chimei lists by chimei category.
chimeis_lists_dict = set_chimeis_lists_dict(filepaths_input_dict)

In [ ]:
def melt_chimei_chimei_df(chimei_chimei_df):
    chimei_chimei_melted_df = pd.melt(
        chimei_chimei_df.reset_index(),
        id_vars='index',
        var_name='chimei2',
        value_name=SIM_NAME
    )
    chimei_chimei_melted_df = chimei_chimei_melted_df.rename(columns={'index': 'chimei1'})
    chimei_chimei_melted_df = chimei_chimei_melted_df.dropna()
        
    # Sort by similarity and reset the index
    chimei_chimei_melted_df = chimei_chimei_melted_df.sort_values(SIM_NAME, ascending=False)
    chimei_chimei_melted_df = chimei_chimei_melted_df.reset_index()
    chimei_chimei_melted_df = chimei_chimei_melted_df.drop(columns='index')
    
    return chimei_chimei_melted_df


### ken - shi/gun/ku/cho/son
def create_df_ken_lower(jppost_df, chimeis_lists_dict, chimei_type_str, chimei_level_float):
    ken_chimei_df = pd.DataFrame()
    ken_list=[]
    for row in chimeis_lists_dict['00_prefectures']:
        for column in chimeis_lists_dict[chimei_type_str]:
            if chimei_level_float == 2.0:
                ken_list = jppost_df[jppost_df['shikuchoson'].str.startswith(column)]['todofuken'].tolist()
            elif chimei_level_float == 2.5:
                ken_shiku_list = jppost_df[jppost_df['concated'].str.endswith('市' + column)]['todofuken'].tolist()
                ken_tokyoku_list = jppost_df[jppost_df['concated'].str.endswith('都' + column)]['todofuken'].tolist()
                ken_list = ken_shiku_list + ken_tokyoku_list
            elif chimei_level_float == 3.0:
                ken_list = jppost_df[jppost_df['shikuchoson'].str.endswith('郡' + column)]['todofuken'].tolist()
            
            if row in ken_list:
                ken_chimei_df.loc[row, column] = 3.0
            else:
                ken_chimei_df.loc[row, column] = 1.0
    
    ken_chimei_melted_df = melt_chimei_chimei_df(ken_chimei_df)
    
    return ken_chimei_df, ken_chimei_melted_df


### shi - shi, gun - gun
def create_df_same_l2_types(chimei_type_str):
    chimei_chimei_df = pd.DataFrame()
    covered_chimei_list = list()
    
    for column in chimeis_lists_dict[chimei_type_str]:
        ken1_set = set(jppost_df[jppost_df['shikuchoson'].str.startswith(column)]['todofuken'].tolist())
        
        for row in chimeis_lists_dict[chimei_type_str]:
            if row not in covered_chimei_list:
                if row == column:
                    chimei_chimei_df.loc[row, column] = 5.0
                else:
                    ken2_set = set(jppost_df[jppost_df['shikuchoson'].str.startswith(row)]['todofuken'].tolist())
                    
                    if len(ken1_set.intersection(ken2_set)) > 0:
                        chimei_chimei_df.loc[row, column] = 2.0
                    else:
                        chimei_chimei_df.loc[row, column] = 1.25
        
        covered_chimei_list.append(column)
    
    chimei_chimei_df = chimei_chimei_df.transpose()
    chimei_chimei_melted_df = melt_chimei_chimei_df(chimei_chimei_df)
    
    return chimei_chimei_df, chimei_chimei_melted_df


### shi - ku, gun - cho/son
def create_df_shiku_gunchoson(jppost_df, chimeis_lists_dict, chimei_type1_str, chimei_type2_str):
    chimei_chimei_df = pd.DataFrame()
    query_str = ""

    for row in chimeis_lists_dict[chimei_type1_str]:
        ken1_set = set(jppost_df[jppost_df['shikuchoson'].str.startswith(row)]['todofuken'].tolist())
        
        for column in chimeis_lists_dict[chimei_type2_str]:
            l2_concat_l3_str = row + column
            
            if chimei_type1_str == '01_shi':
                query_str = '市' + column
            elif chimei_type1_str == '03_gun':
                query_str = '郡' + column
            
            ken2_set = set(jppost_df[jppost_df['shikuchoson'].str.endswith(query_str)]['todofuken'].tolist())
        
            if jppost_df[jppost_df['shikuchoson'].str.contains(l2_concat_l3_str)].size > 0:
                chimei_chimei_df.loc[row, column] = 3.0
            elif len(ken1_set.intersection(ken2_set)) > 0:
                chimei_chimei_df.loc[row, column] = 1.25
            else:
                chimei_chimei_df.loc[row, column] = 1.0
    
    chimei_chimei_melted_df = melt_chimei_chimei_df(chimei_chimei_df)
    
    return chimei_chimei_df, chimei_chimei_melted_df


### shi - cho/son, ku - gun, ku - cho/son
def create_df_for_different_l2s(jppost_df, chimeis_lists_dict, chimei1_type_str, chimei1_level_int, chimei2_type_str):
    chimei_chimei_df = pd.DataFrame()
    covered_chimei_list = list()
    ken1_set = set()
    
    for column in chimeis_lists_dict[chimei1_type_str]:
        # create a todofuken set one of which is the upper level of the column chimei
        if chimei1_level_int == 2:
            ken1_set = set(jppost_df[jppost_df['shikuchoson'].str.startswith(column)]['todofuken'].tolist())
        elif chimei1_level_int == 3:
            ken1_set = set(jppost_df[jppost_df['shikuchoson'].str.endswith(column)]['todofuken'].tolist())
        
        for row in chimeis_lists_dict[chimei2_type_str]:
            # create a todofuken set one of which is the upper level of the row chimei
            ken2_set = set(jppost_df[jppost_df['shikuchoson'].str.endswith(row)]['todofuken'].tolist())
            
            if row not in covered_chimei_list:
                if len(ken1_set.intersection(ken2_set)) > 0:
                    chimei_chimei_df.loc[row, column] = 1.25
                else:
                    chimei_chimei_df.loc[row, column] = 1.0
        
        covered_chimei_list.append(column)
    
    chimei_chimei_df = chimei_chimei_df.transpose()
    chimei_chimei_melted_df = melt_chimei_chimei_df(chimei_chimei_df)
    
    return chimei_chimei_df, chimei_chimei_melted_df


### cho - cho, son - son
def get_upper_levels_of_choson(jppost_df, chimei_name_str):
    gunchoson_df = jppost_df[jppost_df['shikuchoson'].str.endswith('郡' + chimei_name_str)]
    ken_set = set(gunchoson_df['todofuken'].tolist())
    gun_set = set(gunchoson_df['shikuchoson'].replace('(.*)郡(.*)', r'\1郡', regex=True).tolist())
        
    return ken_set, gun_set


def create_df_same_l3_types(jppost_df, chimeis_lists_dict, chimei_type_str):
    chimei_chimei_df = pd.DataFrame()
    covered_chimei_list = list()
    
    for column in chimeis_lists_dict[chimei_type_str]:
        ken1_set, gun1_set = get_upper_levels_of_choson(jppost_df = jppost_df, chimei_name_str = column)
        
        for row in chimeis_lists_dict[chimei_type_str]:
            if row not in covered_chimei_list:
                if row == column:
                    chimei_chimei_df.loc[row, column] = 5.0
                else:
                    ken2_set, gun2_set = get_upper_levels_of_choson(jppost_df = jppost_df, chimei_name_str = row)
                    
                    if len(ken1_set.intersection(ken2_set)) > 0:
                        if len(gun1_set.intersection(gun2_set)) > 0:
                            chimei_chimei_df.loc[row, column] = 2.0
                        else:
                            chimei_chimei_df.loc[row, column] = 1.75
                    else:
                        chimei_chimei_df.loc[row, column] = 1.25
        
        covered_chimei_list.append(column)
    
    chimei_chimei_df = chimei_chimei_df.transpose()
    chimei_chimei_melted_df = melt_chimei_chimei_df(chimei_chimei_df)
    
    return chimei_chimei_df, chimei_chimei_melted_df


In [ ]:
# ken - ken
ken_ken_df = pd.DataFrame()
covered_ken_list = list()

for column in chimeis_lists_dict['00_prefectures']:
    for row in chimeis_lists_dict['00_prefectures']:
        if row not in covered_ken_list:
            if row == column:
                ken_ken_df.loc[row, column] = 5.0
            else:
                ken_ken_df.loc[row, column] = 1.25
    
    covered_ken_list.append(column)

ken_ken_df = ken_ken_df.transpose()
ken_ken_melted_df = melt_chimei_chimei_df(ken_ken_df)

# print(ken_ken_df.head())
# print(ken_ken_df.describe())
ken_ken_melted_df

In [ ]:
# ken - shi
ken_shi_df, ken_shi_melted_df = create_df_ken_lower(
    jppost_df=jppost_df,
    chimeis_lists_dict=chimeis_lists_dict,
    chimei_type_str='01_shi',
    chimei_level_float=2.0
)
print(ken_shi_df.head())
print(ken_shi_df.describe())
ken_shi_melted_df

In [ ]:
# ken - ku
ken_ku_df, ken_ku_melted_df = create_df_ken_lower(
    jppost_df=jppost_df,
    chimeis_lists_dict=chimeis_lists_dict,
    chimei_type_str='02_ku',
    chimei_level_float=2.5
)
print(ken_ku_df.head())
print(ken_ku_df.describe())
ken_ku_melted_df

In [ ]:
# ken - gun
ken_gun_df, ken_gun_melted_df = create_df_ken_lower(
    jppost_df=jppost_df,
    chimeis_lists_dict=chimeis_lists_dict,
    chimei_type_str='03_gun',
    chimei_level_float=2.0
)
print(ken_gun_df.head())
print(ken_gun_df.describe())
ken_gun_melted_df

In [ ]:
# ken - cho
ken_cho_df, ken_cho_melted_df = create_df_ken_lower(
    jppost_df=jppost_df,
    chimeis_lists_dict=chimeis_lists_dict,
    chimei_type_str='04_cho',
    chimei_level_float=3.0
)
# print(ken_cho_df.head())
# print(ken_cho_df.describe())
ken_cho_melted_df

In [ ]:
# ken - son
ken_son_df, ken_son_melted_df = create_df_ken_lower(
    jppost_df=jppost_df,
    chimeis_lists_dict=chimeis_lists_dict,
    chimei_type_str='05_son',
    chimei_level_float=3.0
)
# print(ken_son_df.head())
# print(ken_son_df.describe())
ken_son_melted_df

In [ ]:
# shi - shi
shi_shi_df, shi_shi_melted_df = create_df_same_l2_types(
    chimei_type_str='01_shi'
)
print(shi_shi_df.head())
print(shi_shi_df.describe())
shi_shi_melted_df

In [ ]:
# shi - ku
shi_ku_df, shi_ku_melted_df = create_df_shiku_gunchoson(
    jppost_df=jppost_df,
    chimeis_lists_dict=chimeis_lists_dict,
    chimei_type1_str='01_shi',
    chimei_type2_str='02_ku'
)
print(shi_ku_df.head())
print(shi_ku_df.describe())
shi_ku_melted_df

In [ ]:
# shi - gun
shi_gun_df = pd.DataFrame()

for row in chimeis_lists_dict['01_shi']:
    ken1_set = set(jppost_df[jppost_df['shikuchoson'].str.startswith(row)]['todofuken'].tolist())
    
    for column in chimeis_lists_dict['03_gun']:
        ken2_set = set(jppost_df[jppost_df['shikuchoson'].str.startswith(column)]['todofuken'].tolist())
        
        if len(ken1_set.intersection(ken2_set)) > 0:
            shi_gun_df.loc[row, column] = 1.75
        else:
            shi_gun_df.loc[row, column] = 1.0

shi_gun_melted_df = melt_chimei_chimei_df(shi_gun_df)

print(shi_gun_df.head())
print(shi_gun_df.describe())
shi_gun_melted_df

In [ ]:
# shi - cho
shi_cho_df, shi_cho_melted_df = create_df_for_different_l2s(
    jppost_df=jppost_df,
    chimeis_lists_dict=chimeis_lists_dict,
    chimei1_type_str='01_shi',
    chimei1_level_int=2,
    chimei2_type_str='04_cho'
)
print(shi_cho_df.head())
print(shi_cho_df.describe())
shi_cho_melted_df

In [ ]:
# shi - son
shi_son_df, shi_son_melted_df = create_df_for_different_l2s(
    jppost_df=jppost_df,
    chimeis_lists_dict=chimeis_lists_dict,
    chimei1_type_str='01_shi',
    chimei1_level_int=2,
    chimei2_type_str='05_son'
)
print(shi_son_df.head())
print(shi_son_df.describe())
shi_son_melted_df

In [ ]:
# ku - ku

def categorize_ku(jppost_df, ku_name_str):
    ku_type_str = ''
    
    tokyo_int = jppost_df[jppost_df['shikuchoson'].str.startswith(ku_name_str)].size
    shiku_df = jppost_df[jppost_df['shikuchoson'].str.endswith('市' + ku_name_str)]
    ken_set = set(shiku_df['todofuken'].tolist())
    shi_set = set(shiku_df['shikuchoson'].replace('(.*)市(.*)', r'\1市', regex=True).tolist())
    
    if tokyo_int > 0 and len(ken_set) > 0:
        ku_type_str = 'ts'
    elif tokyo_int > 0 and len(ken_set) == 0:
        ku_type_str = 't'
    elif tokyo_int == 0 and len(ken_set) > 0:
        ku_type_str = 's'
    
    if ku_type_str == '':
        print(f"{ku_name_str}'s type is empty")
    
    return ku_type_str, ken_set, shi_set


ku_ku_df = pd.DataFrame()
covered_ku_list = list()

column_type_str = ''
row_type_str = ''

for column in chimeis_lists_dict['02_ku']:

    column_type_str, ken1_set, shi1_set = categorize_ku(jppost_df = jppost_df, ku_name_str = column)
    
    for row in chimeis_lists_dict['02_ku']:
        
        if row not in covered_ku_list:
            if row == column:
                ku_ku_df.loc[row, column] = 5.0
            else:
                row_type_str, ken2_set, shi2_set = categorize_ku(jppost_df = jppost_df, ku_name_str = row)
                
                ku_types_tuple = (column_type_str, row_type_str)
                
                if ku_types_tuple == ('ts', 'ts'):
                    ku_ku_df.loc[row, column] = 1.75
                elif ku_types_tuple == ('ts', 't') or ku_types_tuple == ('t', 'ts'):
                    ku_ku_df.loc[row, column] = 1.75
                elif ku_types_tuple == ('ts', 's') or ku_types_tuple == ('s', 'ts'):
                    if len(ken1_set.intersection(ken2_set)) > 0:
                        ku_ku_df.loc[row, column] = 1.75
                    else:
                        ku_ku_df.loc[row, column] = 1.25
                elif ku_types_tuple == ('t', 't'):
                    ku_ku_df.loc[row, column] = 1.75
                elif ku_types_tuple == ('t', 's') or ku_types_tuple == ('s', 't'):
                    ku_ku_df.loc[row, column] = 1.25
                elif ku_types_tuple == ('s', 's'):
                    if len(ken1_set.intersection(ken2_set)) > 0:
                        if len(shi1_set.intersection(shi2_set)) > 0:
                            ku_ku_df.loc[row, column] = 2.0
                        else:
                            ku_ku_df.loc[row, column] = 1.75
                    else:
                        ku_ku_df.loc[row, column] = 1.25

    covered_ku_list.append(column)

ku_ku_df = ku_ku_df.transpose()
ku_ku_melted_df = melt_chimei_chimei_df(ku_ku_df)

# print(ku_ku_df.head())
# print(ku_ku_df.describe())
ku_ku_melted_df

In [ ]:
# ku - gun
ku_gun_df, ku_gun_melted_df = create_df_for_different_l2s(
    jppost_df=jppost_df,
    chimeis_lists_dict=chimeis_lists_dict,
    chimei1_type_str='03_gun',
    chimei1_level_int=2,
    chimei2_type_str='02_ku'
)
print(ku_gun_df.head())
print(ku_gun_df.describe())
ku_gun_melted_df

In [ ]:
# ku - cho
ku_cho_df, ku_cho_melted_df = create_df_for_different_l2s(
    jppost_df=jppost_df,
    chimeis_lists_dict=chimeis_lists_dict,
    chimei1_type_str='02_ku',
    chimei1_level_int=3,
    chimei2_type_str='04_cho'
)
print(ku_cho_df.head())
print(ku_cho_df.describe())
ku_cho_melted_df

In [ ]:
# ku - son
ku_son_df, ku_son_melted_df = create_df_for_different_l2s(
    jppost_df=jppost_df,
    chimeis_lists_dict=chimeis_lists_dict,
    chimei1_type_str='02_ku',
    chimei1_level_int=3,
    chimei2_type_str='05_son'
)
print(ku_son_df.head())
print(ku_son_df.describe())
ku_son_melted_df

In [ ]:
# gun - gun
gun_gun_df, gun_gun_melted_df = create_df_same_l2_types(
    chimei_type_str='03_gun'
)
print(gun_gun_df.head())
print(gun_gun_df.describe())
gun_gun_melted_df

In [ ]:
# gun - cho
gun_cho_df, gun_cho_melted_df = create_df_shiku_gunchoson(
    jppost_df=jppost_df,
    chimeis_lists_dict=chimeis_lists_dict,
    chimei_type1_str='03_gun',
    chimei_type2_str='04_cho'
)
print(gun_cho_df.head())
print(gun_cho_df.describe())
gun_cho_melted_df

In [ ]:
# gun - son
gun_son_df, gun_son_melted_df = create_df_shiku_gunchoson(
    jppost_df=jppost_df,
    chimeis_lists_dict=chimeis_lists_dict,
    chimei_type1_str='03_gun',
    chimei_type2_str='05_son'
)
print(gun_son_df.head())
print(gun_son_df.describe())
gun_son_melted_df

In [ ]:
# cho - cho
cho_cho_df, cho_cho_melted_df = create_df_same_l3_types(
    jppost_df=jppost_df,
    chimeis_lists_dict=chimeis_lists_dict,
    chimei_type_str='04_cho'
)
print(cho_cho_df.head())
print(cho_cho_df.describe())
cho_cho_melted_df

In [ ]:
# cho - son
cho_son_df = pd.DataFrame()

for column in chimeis_lists_dict['04_cho']:
    ken1_set, gun1_set = get_upper_levels_of_choson(jppost_df=jppost_df, chimei_name_str=column)

    for row in chimeis_lists_dict['05_son']:
        ken2_set, gun2_set = get_upper_levels_of_choson(jppost_df=jppost_df, chimei_name_str=row)
        
        if len(ken1_set.intersection(ken2_set)) > 0:
            if len(gun1_set.intersection(gun2_set)) > 0:
                cho_son_df.loc[row, column] = 1.75
            else:
                cho_son_df.loc[row, column] = 1.25
        else:
            cho_son_df.loc[row, column] = 1.0

cho_son_melted_df = melt_chimei_chimei_df(cho_son_df)

print(cho_son_df.head())
print(cho_son_df.describe())
cho_son_melted_df

In [ ]:
# son - son
son_son_df, son_son_melted_df = create_df_same_l3_types(
    jppost_df=jppost_df,
    chimeis_lists_dict=chimeis_lists_dict,
    chimei_type_str='05_son'
)
print(son_son_df.head())
print(son_son_df.describe())
son_son_melted_df

In [ ]:
chimei_chimei_dfs_list = [
    ken_ken_melted_df,
    ken_shi_melted_df,
    ken_ku_melted_df,
    ken_gun_melted_df,
    ken_cho_melted_df,
    ken_son_melted_df,
    shi_shi_melted_df,
    shi_ku_melted_df,
    shi_gun_melted_df,
    shi_cho_melted_df,
    shi_son_melted_df,
    ku_ku_melted_df,
    ku_gun_melted_df,
    ku_cho_melted_df,
    ku_son_melted_df,
    gun_gun_melted_df,
    gun_cho_melted_df,
    gun_son_melted_df,
    cho_cho_melted_df,
    cho_son_melted_df,
    son_son_melted_df
]
chimei_pairs_df = pd.concat(chimei_chimei_dfs_list, axis=0, ignore_index=True)
chimei_pairs_df

In [ ]:
filepath_to_save_str = 'sts_chimei_pairs/sts_chimei_pairs.csv'
chimei_pairs_df.to_csv(filepath_to_save_str)